# สแกนรูปงานโรงเรียน → ดัชนีใบหน้า (สำหรับเว็บ "ค้นหารูปของฉัน")

รันทีละเซลล์จากบนลงล่าง ใช้เวลาครั้งแรกตามจำนวนรูป ครั้งต่อไปสแกนเฉพาะรูปใหม่

โครงโฟลเดอร์รูปใน Drive: `ราก / ปี พ.ศ. / เดือน / ชื่องาน / รูป...` (โฟลเดอร์ที่ขึ้นต้นด้วย `_` ถูกข้าม)

## 1) ติดตั้งเครื่องมือ

In [ ]:
!pip -q install playwright pillow pillow-heif google-api-python-client
!playwright install --with-deps chromium

## 2) อัปโหลดไฟล์ zip ของโปรเจกต์เว็บ (ไฟล์เดียวกับที่ใช้ขึ้นเว็บ)

In [ ]:
from google.colab import files
import zipfile, os, glob
up = files.upload()
zname = next(n for n in up if n.lower().endswith('.zip'))
zipfile.ZipFile(zname).extractall('/content/site')
SITE = os.path.dirname(glob.glob('/content/site/**/face-api.js', recursive=True)[0])
print('โฟลเดอร์เว็บ:', SITE)

## 3) ตั้งค่า และเข้าสู่ระบบ Google

- `ROOT_FOLDER_ID` = ID โฟลเดอร์รากของรูป **หรือ ID โฟลเดอร์ปี** (เช่น `2569`) ใส่หลายอันได้โดยคั่นด้วย `,` เช่น `id2569,id2570` (ส่วนท้ายลิงก์ `.../folders/<ID>`)
- `INDEX_FOLDER_ID` = ID โฟลเดอร์ `_faceindex` ของคุณ (แนะนำให้ใส่ ถ้าไม่มีสิทธิ์แก้ไขโฟลเดอร์รูป) ปล่อยว่างได้ถ้ามีสิทธิ์ ระบบจะหา/สร้างให้ใต้โฟลเดอร์แรก

In [ ]:
ROOT_FOLDER_ID = ''    # <- ID โฟลเดอร์ราก หรือโฟลเดอร์ปี (หลายอันคั่นด้วย ,)
INDEX_FOLDER_ID = ''   # <- (ไม่บังคับ) ID โฟลเดอร์เก็บดัชนี

from google.colab import auth
auth.authenticate_user()

import subprocess, sys
def run_indexer(*extra):
    cmd = [sys.executable, f'{SITE}/tools/indexer.py', '--source', 'drive', '--root', ROOT_FOLDER_ID, '--site', SITE, *extra]
    if INDEX_FOLDER_ID: cmd += ['--index-folder', INDEX_FOLDER_ID]
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout: print(line, end='')
    p.wait()

## 4) ตรวจก่อนว่าระบบอ่านโฟลเดอร์ปี/เดือน/งาน ได้ถูก (ยังไม่สแกน)

In [ ]:
run_indexer('--list-only')

## 5) สแกนจริง (รันซ้ำได้ ระบบข้ามรูปที่สแกนแล้ว)

In [ ]:
run_indexer()

## 6) หลังสแกนเสร็จ

1. ใน Drive เปิดโฟลเดอร์ `_faceindex` → แชร์ → **ทุกคนที่มีลิงก์ = ผู้ดู** (ถ้ายังไม่ได้แชร์)
2. คัดลอก ID ของโฟลเดอร์ `_faceindex` ไปใส่ `driveFolderId` ใน `config.js` ของเว็บ
3. มีงานใหม่เมื่อไร เพิ่มรูปในโฟลเดอร์งาน แล้วรันเซลล์ที่ 5 ใหม่ เว็บจะเห็นงานใหม่เอง ไม่ต้องแก้เว็บ